# OpenPlaque — Secondary Branch Rejection Diagnostic
Instrumented target-free continuation diagnostic. Uses unchanged thresholds and records why every attempted next step is rejected. Research use only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse controls: True=reuse valid cache; False=force recompute.
REUSE_SOURCE_CT = True
REUSE_FROZEN_GEOMETRY = True
REUSE_DIAGNOSTIC_SEARCH = False

In [ ]:
# Dependencies
!pip -q install scipy pandas matplotlib

In [ ]:
# Clone the fresh experiment branch
import os, shutil
if os.path.exists('/content/OpenPlaque'):
    shutil.rmtree('/content/OpenPlaque')
!git clone -q --depth 1 --branch secondary-rejection-diagnostics-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%cd /content/OpenPlaque

In [ ]:
# Initialize workflow
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.secondary_rejection_diagnostic import SecondaryRejectionDiagnosticWorkflow
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'frozen_geometry': REUSE_FROZEN_GEOMETRY,
    'diagnostic_search': REUSE_DIAGNOSTIC_SEARCH,
}
wf = SecondaryRejectionDiagnosticWorkflow(reuse=reuse)
display(wf.cache_status())

In [ ]:
# Load source CCTA and freeze the same target-free source geometry
wf.load_source_ct()
geometry = wf.load_frozen_geometry()
display(geometry)

In [ ]:
# Run rejection-reason diagnostic with unchanged thresholds
summary = wf.run_diagnostic(max_new_mm=4.0, beam_width=50)
display(summary)
display(wf.rejection_summary)
if summary.get('first_dead_step') is not None:
    terminal = wf.attempts[wf.attempts.step_index == summary['first_dead_step']]
    display(terminal['rejection_reason'].value_counts().rename('count').to_frame())
    cols = [c for c in ['rejection_reason','nearest_component_radius_mm','radius_mm','recenter_shift_mm','plane_score','actual_turn_deg','old_branch_separation_mm','self_separation_mm','tortuosity'] if c in terminal.columns]
    display(terminal[cols].head(30))

In [ ]:
# Figures
figures = wf.make_figures()
from IPython.display import display, Image
for p in figures:
    display(Image(filename=str(p)))

In [ ]:
# Package final report back to Drive
from urllib.parse import quote
report = wf.make_report()
zip_path = wf.package()
final_name = zip_path.name
drive_search = 'https://drive.google.com/drive/u/0/search?q=' + quote(final_name)
print('Report:', report)
print('Report-back ZIP:', zip_path)
print('Direct Drive search link:', drive_search)